In [1]:
# Complete Multimodal Price Prediction with Text + Images
# Target: Achieve lowest possible SMAPE

import os
import pandas as pd
import numpy as np
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from scipy.sparse import hstack, csr_matrix
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

class MultimodalPricePredictor:
    def __init__(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.setup_image_model()
        
    def setup_image_model(self):
        """Initialize image feature extractor"""
        self.image_model = models.resnet50(pretrained=True)
        self.image_model = nn.Sequential(*list(self.image_model.children())[:-1])  # Remove classification layer
        self.image_model.eval()
        self.image_model.to(self.device)
        
        self.image_transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    
    def extract_image_features(self, image_paths):
        """Extract ResNet features from images"""
        features = []
        for path in tqdm(image_paths, desc="Extracting image features"):
            try:
                if os.path.exists(path):
                    img = Image.open(path).convert('RGB')
                    img_t = self.image_transform(img).unsqueeze(0).to(self.device)
                    with torch.no_grad():
                        feat = self.image_model(img_t).squeeze().cpu().numpy()
                    features.append(feat.flatten())
                else:
                    features.append(np.zeros(2048))  # ResNet50 feature size
            except Exception as e:
                features.append(np.zeros(2048))
                print(f"Error processing {path}: {e}")
        return np.array(features)
    
    def extract_value_unit_pack(self, text):
        """Enhanced feature extraction from catalog content"""
        s = str(text)
        
        # Value extraction with multiple patterns
        v = None
        patterns = [
            r'Value:\s*([\d,.]+)',
            r'Size:\s*([\d,.]+)',
            r'(\d+(?:\.\d+)?)\s*(?:oz|ounce|fl oz|ml|g|gram|kg|pound|lb)\b',
            r'(\d+(?:\.\d+)?)\s*[x×]\s*\d+'  # 2x100ml pattern
        ]
        
        for pattern in patterns:
            m = re.search(pattern, s, flags=re.IGNORECASE)
            if m:
                try:
                    v = float(m.group(1).replace(',', ''))
                    break
                except:
                    continue
        
        # Unit extraction
        unit = "Unknown"
        unit_patterns = [
            r'Unit:\s*([A-Za-z0-9 %\.\-/]+)',
            r'\b(oz|ounce|fl oz|fluid ounce|ml|milliliter|g|gram|kg|kilogram|lb|pound|count|piece|tablet|capsule)\b'
        ]
        
        for pattern in unit_patterns:
            m = re.search(pattern, s, flags=re.IGNORECASE)
            if m:
                unit = m.group(1).strip()
                break
        
        # Pack extraction
        pack = 1
        pack_patterns = [
            r'Pack(?:\s*of)?\s*(\d+)',
            r'(\d+)\s*[cC]ount',
            r'(\d+)\s*[pP]ack',
            r'case\s*of\s*(\d+)',
            r'(\d+)\s*[x×]'
        ]
        
        for pattern in pack_patterns:
            m = re.search(pattern, s, flags=re.IGNORECASE)
            if m:
                try:
                    pack = int(m.group(1))
                    break
                except:
                    continue
        
        return v, unit, pack
    
    def advanced_unit_conversion(self, value, unit):
        """Improved unit conversion with separate weight/volume handling"""
        if value is None or pd.isna(value):
            return np.nan, "unknown"
            
        u = str(unit).lower()
        v = float(value)
        unit_type = "unknown"
        
        # Weight units
        if re.search(r'gram|g\b', u) and not re.search(r'program', u):
            converted = v / 28.35  # g -> oz
            unit_type = "weight"
        elif re.search(r'kg\b|kilogram', u):
            converted = (v * 1000) / 28.35  # kg -> g -> oz
            unit_type = "weight"
        elif re.search(r'lb\b|pound', u):
            converted = v * 16.0  # lb -> oz
            unit_type = "weight"
        elif re.search(r'ounce|oz\b', u) and not re.search(r'fl oz|fluid', u):
            converted = v
            unit_type = "weight"
        
        # Volume units (keep separate)
        elif re.search(r'fl oz|floz|fl\. oz|fluid ounce', u):
            converted = v
            unit_type = "volume"
        elif re.search(r'ml\b|milliliter', u):
            converted = v / 29.57  # ml -> fl oz
            unit_type = "volume"
        
        # Count units
        elif re.search(r'count|piece|pcs|packets|tablet|capsule|caps|ct', u):
            converted = v
            unit_type = "count"
        
        else:
            converted = v
            unit_type = "unknown"
        
        return converted, unit_type
    
    def extract_brand_category(self, text):
        """Extract brand and category information"""
        s = str(text).lower()
        
        # Common brands pattern
        brands = ['amazon', 'apple', 'samsung', 'sony', 'nike', 'adidas', 'dell', 'hp', 'lenovo', 
                 'colgate', 'gillette', 'pampers', 'huggies', 'nestle', 'kraft']
        
        brand = "unknown"
        for b in brands:
            if b in s:
                brand = b
                break
        
        # Category patterns
        category = "unknown"
        categories = {
            'electronics': r'\b(tv|television|phone|smartphone|laptop|tablet|computer|camera|headphone|earphone)\b',
            'clothing': r'\b(shirt|pants|jeans|dress|shoes|sneakers|jacket|hoodie)\b',
            'food': r'\b(food|snack|chocolate|coffee|tea|cereal|pasta|sauce)\b',
            'beauty': r'\b(shampoo|conditioner|soap|cream|lotion|makeup|cosmetic|perfume)\b',
            'baby': r'\b(diaper|baby|infant|toy|pacifier|bottle)\b',
            'home': r'\b(furniture|chair|table|bed|mattress|pillow|blanket)\b'
        }
        
        for cat, pattern in categories.items():
            if re.search(pattern, s):
                category = cat
                break
        
        return brand, category
    
    def create_advanced_features(self, df, image_dir):
        """Create comprehensive feature set"""
        print("Extracting text features...")
        
        # Basic features
        features = []
        brands = []
        categories = []
        unit_types = []
        
        for idx, row in tqdm(df.iterrows(), total=len(df)):
            catalog_content = row['catalog_content']
            
            # Extract basic features
            value, unit, pack = self.extract_value_unit_pack(catalog_content)
            value_base, unit_type = self.advanced_unit_conversion(value, unit)
            total_qty = value_base * pack if value_base else 0
            
            # Extract brand and category
            brand, category = self.extract_brand_category(catalog_content)
            
            features.append({
                'Value': value if value else 0,
                'Unit': unit,
                'Pack': pack,
                'Value_base': value_base if value_base else 0,
                'Total_Qty': total_qty,
                'text_length': len(str(catalog_content))
            })
            brands.append(brand)
            categories.append(category)
            unit_types.append(unit_type)
        
        # Create feature DataFrame
        feature_df = pd.DataFrame(features)
        feature_df['brand'] = brands
        feature_df['category'] = categories
        feature_df['unit_type'] = unit_types
        
        # One-hot encode categorical features
        categorical_cols = ['brand', 'category', 'unit_type']
        for col in categorical_cols:
            dummies = pd.get_dummies(feature_df[col], prefix=col)
            feature_df = pd.concat([feature_df, dummies], axis=1)
        
        # Drop original categorical columns
        feature_df = feature_df.drop(categorical_cols + ['Unit'], axis=1)
        
        # Extract image features
        print("Extracting image features...")
        image_paths = []
        for idx, row in df.iterrows():
            image_url = row['image_link']
            if pd.notna(image_url):
                filename = image_url.split('/')[-1]
                # Try different possible locations
                possible_paths = [
                    os.path.join(image_dir, 'train', filename),
                    os.path.join(image_dir, 'test', filename),
                    os.path.join(image_dir, filename),
                    os.path.join('images', 'train', filename),
                    os.path.join('images', 'test', filename),
                    os.path.join('images', filename)
                ]
                found = False
                for path in possible_paths:
                    if os.path.exists(path):
                        image_paths.append(path)
                        found = True
                        break
                if not found:
                    image_paths.append('')  # Mark as missing
            else:
                image_paths.append('')
        
        image_features = self.extract_image_features(image_paths)
        
        # TF-IDF on text
        print("Creating TF-IDF features...")
        self.tfidf = TfidfVectorizer(
            max_features=3000, 
            stop_words='english', 
            ngram_range=(1, 2),
            min_df=2
        )
        
        if hasattr(self, 'tfidf_fitted'):
            text_features = self.tfidf.transform(df['catalog_content'].fillna(''))
        else:
            text_features = self.tfidf.fit_transform(df['catalog_content'].fillna(''))
            self.tfidf_fitted = True
        
        # Combine all features
        print("Combining all features...")
        numeric_features = feature_df.values
        all_features = hstack([
            text_features,
            csr_matrix(numeric_features),
            csr_matrix(image_features)
        ])
        
        return all_features, feature_df.columns.tolist()
    
    def train_model(self, train_df, test_df, image_dir):
        """Train the complete multimodal model"""
        print("Creating training features...")
        X_train, feature_names = self.create_advanced_features(train_df, image_dir)
        y_train = np.log1p(train_df['price'].values)  # Log transform
        
        print("Creating test features...")
        X_test, _ = self.create_advanced_features(test_df, image_dir)
        
        print(f"Final feature dimensions - Train: {X_train.shape}, Test: {X_test.shape}")
        
        # Train LightGBM with optimized parameters
        print("Training LightGBM model...")
        train_data = lgb.Dataset(X_train, label=y_train)
        
        params = {
            'objective': 'regression',
            'metric': 'l2',
            'learning_rate': 0.02,  # Lower for better convergence
            'num_leaves': 128,      # More leaves for complex patterns
            'max_depth': -1,        # Unlimited depth
            'min_data_in_leaf': 20,
            'feature_fraction': 0.8,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'lambda_l1': 0.1,
            'lambda_l2': 0.1,
            'verbosity': -1,
            'seed': SEED,
            'n_jobs': -1
        }
        
        self.model = lgb.train(
            params,
            train_data,
            num_boost_round=2000,
            valid_sets=[train_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=True),
                lgb.log_evaluation(100)
            ]
        )
        
        return X_test
    
    def predict(self, X_test):
        """Make predictions"""
        y_pred_log = self.model.predict(X_test, num_iteration=self.model.best_iteration)
        y_pred = np.expm1(y_pred_log)  # Convert back from log
        y_pred = np.clip(y_pred, 0.1, None)  # Ensure positive prices
        return y_pred

def calculate_smape(y_true, y_pred):
    """Calculate SMAPE metric"""
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    denominator = np.where(denominator == 0, 1.0, denominator)  # Avoid division by zero
    return np.mean(np.abs(y_pred - y_true) / denominator) * 100.0

def main():
    # Configuration
    DATASET_FOLDER = 'D:/Projects/Amazon ML Challenge/68e8d1d70b66d_student_resource/student_resource/faster_image_pricing/faster_image_pricing_mm/dataset'
    IMAGE_DIR = 'D:/Projects/Amazon ML Challenge/68e8d1d70b66d_student_resource/student_resource/faster_image_pricing/faster_image_pricing_mm/images'  # Update this path
    OUTPUT_FOLDER = './outputs'
    
    # Create output directory
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    
    # Load data
    print("Loading data...")
    train_df = pd.read_csv(os.path.join(DATASET_FOLDER, 'train.csv'))
    test_df = pd.read_csv(os.path.join(DATASET_FOLDER, 'test.csv'))
    
    print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")
    
    # Initialize predictor
    predictor = MultimodalPricePredictor()
    
    # Train model and get test features
    X_test = predictor.train_model(train_df, test_df, IMAGE_DIR)
    
    # Make predictions
    print("Making predictions...")
    predictions = predictor.predict(X_test)
    
    # Create output
    output_df = pd.DataFrame({
        'sample_id': test_df['sample_id'],
        'price': predictions
    })
    
    # Save predictions
    output_path = os.path.join(OUTPUT_FOLDER, 'test_out_multimodal.csv')
    output_df.to_csv(output_path, index=False)
    
    print(f"Predictions saved to: {output_path}")
    print(f"Sample predictions:\n{output_df.head(10)}")
    
    # If you want to calculate validation score (optional)
    from sklearn.model_selection import train_test_split
    print("\nCalculating validation score...")
    
    # Create validation split
    X_train_full, _ = predictor.create_advanced_features(train_df, IMAGE_DIR)
    y_train_full = np.log1p(train_df['price'].values)
    
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=SEED
    )
    
    # Train validation model
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    params = {
        'objective': 'regression',
        'metric': 'l2',
        'learning_rate': 0.02,
        'num_leaves': 128,
        'verbosity': -1,
        'seed': SEED
    }
    
    val_model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(100)]
    )
    
    # Validate
    y_val_pred_log = val_model.predict(X_val, num_iteration=val_model.best_iteration)
    y_val_pred = np.expm1(y_val_pred_log)
    y_val_true = np.expm1(y_val)
    
    val_smape = calculate_smape(y_val_true, y_val_pred)
    print(f"Validation SMAPE: {val_smape:.4f}%")
    
    # Save model for future use
    model_path = os.path.join(OUTPUT_FOLDER, 'multimodal_model.txt')
    predictor.model.save_model(model_path)
    print(f"Model saved to: {model_path}")

if __name__ == "__main__":
    main()

Loading data...
Train shape: (75000, 4), Test shape: (75000, 3)
Creating training features...
Extracting text features...


100%|██████████████████████████████████████████████████████████████████████████| 75000/75000 [00:14<00:00, 5279.14it/s]


Extracting image features...


Extracting image features:  93%|███████████████████████████████████████████▉   | 70074/75000 [2:17:44<20:09,  4.07it/s]

Error processing D:/Projects/Amazon ML Challenge/68e8d1d70b66d_student_resource/student_resource/faster_image_pricing/faster_image_pricing_mm/images\train\71xpOK8TVqL.jpg: image file is truncated (35 bytes not processed)


Extracting image features: 100%|███████████████████████████████████████████████| 75000/75000 [2:42:49<00:00,  7.68it/s]


Creating TF-IDF features...
Combining all features...


ValueError: scipy.sparse does not support dtype object. The only supported types are: bool, int8, uint8, int16, uint16, intc, uintc, int32, uint32, int64, uint64, float32, float64, longdouble, complex64, complex128, clongdouble.